# Tahap 6 — Decision-Support Scenario Evaluation

Notebook ini mengevaluasi rule decision support yang transparan berdasarkan Canonical Twin State, occupancy, current power, forecast power 30 menit, temperature, dan humidity. Evaluasi ini bukan pengujian energy saving, causal impact, efektivitas manusia, atau autonomous control.

In [ ]:
# 2. Environment check
from pathlib import Path
import importlib.metadata
import os
import platform
import sys

REPOSITORY_URL = 'https://github.com/rehanalfarizu/new_jurnal.git'
IS_COLAB = 'google.colab' in sys.modules

def find_repo_root(start):
    current = Path(start).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'configs' / 'experiment.yaml').is_file() and (candidate / 'src').is_dir():
            return candidate
    return None

REPO_ROOT = find_repo_root(Path.cwd())
print({'python': sys.version.split()[0], 'platform': platform.platform(), 'colab': IS_COLAB, 'repo_root': str(REPO_ROOT) if REPO_ROOT else None})

In [ ]:
# 3. Google Colab setup
import subprocess

if IS_COLAB and REPO_ROOT is None:
    clone_target = Path('/content/new_jurnal')
    if not clone_target.exists():
        subprocess.run(['git', 'clone', REPOSITORY_URL, str(clone_target)], check=True)
    REPO_ROOT = clone_target
if REPO_ROOT is None:
    raise RuntimeError('Root repository new_jurnal tidak ditemukan.')
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print('Working directory:', Path.cwd())

In [ ]:
# 4. Dependency setup
import importlib.util

required_modules = ['pandas', 'yaml', 'matplotlib']
missing_modules = [name for name in required_modules if importlib.util.find_spec(name) is None]
if missing_modules and IS_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)
elif missing_modules:
    raise RuntimeError(f'Dependency belum tersedia: {missing_modules}. Jalankan pip install -r requirements.txt')
print('Dependency minimum tersedia.')

In [ ]:
# 5. Dataset/results integrity
import hashlib
import json

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

PREDICTIONS_PATH = Path(os.environ.get('FORECAST_PREDICTIONS_PATH', 'results/metrics/occupancy_ablation_test_predictions.csv'))
MODELING_STATE_PATH = Path(os.environ.get('MODELING_STATE_PATH', 'data/processed/modeling_1min.csv'))
for required_path in [PREDICTIONS_PATH, MODELING_STATE_PATH, Path('results/metrics/occupancy_ablation_manifest.json')]:
    if not required_path.is_file():
        raise FileNotFoundError(f'Artefak wajib tidak ditemukan: {required_path}. Atur FORECAST_PREDICTIONS_PATH atau MODELING_STATE_PATH bila disimpan di lokasi lain.')
print({'predictions_sha256': sha256_file(PREDICTIONS_PATH), 'modeling_state_sha256': sha256_file(MODELING_STATE_PATH)})

In [ ]:
# 6. Load Canonical State config
import pandas as pd
import yaml
from IPython.display import display

with Path('configs/experiment.yaml').open(encoding='utf-8') as handle:
    experiment_config = yaml.safe_load(handle)
with Path('configs/decision_support.yaml').open(encoding='utf-8') as handle:
    decision_config = yaml.safe_load(handle)
display(pd.DataFrame([experiment_config['canonical_state']]).drop(columns=['validation_ranges']))
print('room_id dipertahankan:', experiment_config['canonical_state']['room_id'])

In [ ]:
# 7. Load forecasting outputs
forecast_output = pd.read_csv(PREDICTIONS_PATH)
with Path('results/metrics/occupancy_ablation_manifest.json').open(encoding='utf-8') as handle:
    stage4_manifest = json.load(handle)
display(forecast_output.head())
print({'rows': len(forecast_output), 'split': sorted(forecast_output['split'].unique()), 'forecast_model': decision_config['input']['forecast_model'], 'model_retrained_here': False})

In [ ]:
# 8. Decision-support inputs
from src.decision_support.evaluation import load_decision_support_inputs

decision_inputs, input_metadata = load_decision_support_inputs(
    predictions_path=PREDICTIONS_PATH,
    modeling_path=MODELING_STATE_PATH,
    experiment_config_path='configs/experiment.yaml',
    decision_config=decision_config,
    device_summary_path='results/tables/device_summary.csv',
)
display(decision_inputs.head())
print(input_metadata)

In [ ]:
# 9. Rule definitions
rule_rows = []
for rule in decision_config['rules'].values():
    rule_rows.append({
        'rule_id': rule['rule_id'],
        'name': rule['name'],
        'severity': rule['severity'],
        'threshold_basis': rule['threshold_basis'],
        'configuration': json.dumps(rule, ensure_ascii=False, sort_keys=True),
    })
display(pd.DataFrame(rule_rows))
print('Threshold numerik adalah declared research scenario threshold, bukan standard comfort atau safety limit.')

In [ ]:
# 10. Example scenario trace
from src.decision_support.engine import DecisionSupportEngine, DecisionSupportInput

engine = DecisionSupportEngine(decision_config)
example = decision_inputs.iloc[0]
example_state = DecisionSupportInput(
    timestamp_utc=example['timestamp_utc'].isoformat(),
    occupancy_count=int(example['occupancy_count']),
    current_power_w=float(example['current_power_w']),
    forecast_power_30m_w=float(example['forecast_power_30m_w']),
    temperature_c=float(example['temperature_c']),
    humidity_percent=float(example['humidity_percent']),
    device_id=example['device_id'], room_id=example['room_id'], schema_version=example['schema_version'],
)
display(pd.json_normalize(engine.evaluate(example_state)))

In [ ]:
# 11. Run evaluation
from src.decision_support.evaluation import run_decision_support_evaluation

manifest = run_decision_support_evaluation(
    predictions_path=PREDICTIONS_PATH,
    modeling_path=MODELING_STATE_PATH,
)
display(pd.DataFrame([manifest['summary']]).drop(columns=['recommendation_count_by_rule']))

In [ ]:
# 12. Recommendation distribution
scenarios = pd.read_csv('results/tables/decision_support_scenarios.csv')
rule_summary = pd.read_csv('results/tables/decision_support_rule_summary.csv')
display(rule_summary[['rule_id', 'rule_name', 'recommendation_count', 'sample_coverage', 'threshold_basis']])

In [ ]:
# 13. Rule summary
display(rule_summary[['rule_id', 'action_class', 'severity', 'priority', 'threshold_configuration_json', 'recommendation_text']])
print('Jumlah recommendation:', manifest['summary']['recommendation_count'])
print('Coverage active recommendation:', manifest['summary']['recommendation_coverage'])
print('No-action count:', manifest['summary']['no_action_count'])

In [ ]:
# 14. Occupancy-level analysis
occupancy_distribution = pd.DataFrame.from_dict(manifest['occupancy_distribution'], orient='index')
occupancy_distribution.index.name = 'occupancy_count'
display(occupancy_distribution)
print('Distribusi dihitung dari sample aktual, bukan outcome sintetis.')

In [ ]:
# 15. Forecast-delta analysis
from IPython.display import Image

recomputed_delta = scenarios['forecast_power_30m_w'] - scenarios['current_power_w']
if not (recomputed_delta - scenarios['forecast_delta_w']).abs().lt(1e-12).all():
    raise AssertionError('forecast_delta_w tidak konsisten dengan definisi forecast - current.')
display(scenarios.groupby('rule_id')['forecast_delta_w'].describe())
display(Image(filename='results/figures/decision_support_forecast_delta.png'))

In [ ]:
# 16. Consistency/determinism
consistency = pd.read_csv('results/tables/decision_support_consistency.csv')
display(consistency)
if not (consistency['status'] == 'PASS').all():
    raise AssertionError('Terdapat consistency check yang gagal.')
for figure_path in [
    'results/figures/decision_support_rule_frequency.png',
    'results/figures/decision_support_occupancy_distribution.png',
]:
    display(Image(filename=figure_path))

In [ ]:
# 17. Limitations
from IPython.display import Markdown

display(Markdown('### Keterbatasan interpretasi\n\n' + '\n'.join(f'- {item}' for item in manifest['limitations'])))

In [ ]:
# 18. Reproducibility summary
runtime_packages = {}
for package in ['numpy', 'pandas', 'matplotlib', 'PyYAML']:
    try:
        runtime_packages[package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        runtime_packages[package] = None
reproducibility = {
    'evaluation_name': manifest['evaluation_name'],
    'input_sha256': manifest['input_sha256'],
    'code_sha256': manifest['code_sha256'],
    'git_commit': manifest.get('git_commit'),
    'runtime_packages': runtime_packages,
    'all_consistency_checks_passed': manifest['summary']['all_consistency_checks_passed'],
}
display(reproducibility)